# VACV SI: comparison of reproducible experiments

Read-only report: compare saved OOF and verified Kaggle Scores. No model fitting, test endpoint access or uploads.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
root = Path.cwd().resolve()
while not (root / 'src/current/data.py').exists():
    if root.parent == root: raise RuntimeError('Run inside the repository')
    root = root.parent
sys.path.insert(0, str(root))
from src.current.scoreboard import build
table = build(write=False)
print(f'{len(table)} evaluated configurations; identical 267 development rows and 18 folds')


132 evaluated configurations; identical 267 development rows and 18 folds


## Primary raw-SI OOF MSE

Includes zeros and every original development row. This is the competition formula on training OOF, not the hidden Kaggle test score. Broad search reuses OOF; there is no independent holdout confirmation.

In [2]:
columns = ['experiment','candidate','mean_oof_mse','zero_mse','bulk_mse','row14_loss_share']
print(table[columns].head(10).round(2).to_string(index=False))


                experiment                                        candidate  mean_oof_mse  zero_mse  bulk_mse  row14_loss_share
current_probability_public              gp_power2.0_rho1.0_observation_mean    4973199.16  15944.26 136661.05              0.96
current_probability_public                   gp_power2.0_rho1.0_latent_mean    4977286.33  10917.88  94907.14              0.96
 current_representation_gp      rbf_tanimoto_public_plus_kaggle_latent_mean    4986876.20  13071.29  73983.59              0.96
         current_hurdle_gp               2.0_public_plus_kaggle_latent_mean    5002853.26  21527.97 155442.23              0.96
current_probability_public              gp_power1.0_rho0.5_observation_mean    5010481.21  11978.30  75619.67              0.95
 current_representation_gp rbf_tanimoto_public_plus_kaggle_observation_mean    5013150.34  18973.45 101808.45              0.95
current_probability_public              gp_power2.0_rho0.5_observation_mean    5014082.10   5815.27  674

## Sensitivity to the single extreme observation

Row14 has SI=36066.667 and accounts for about 96% of some candidates' OOF loss. The following diagnostic does not delete it, change labels or redefine the official metric.

In [3]:
print(table.sort_values('without_row14_mse')[columns[:2]+['without_row14_mse','mean_oof_mse']].head(10).round(2).to_string(index=False))


                experiment                                   candidate  without_row14_mse  mean_oof_mse
current_probability_public              gp_power2.0_rho1.0_latent_mean          190426.60    4977286.33
 current_representation_gp rbf_tanimoto_public_plus_kaggle_latent_mean          197499.56    4986876.20
current_probability_public         gp_power2.0_rho1.0_observation_mean          202861.71    4973199.16
      current_bio_transfer                         bio_fp__kaggle_only          207208.65    5014266.30
         current_bio_blend                             bio_weight_0.75          208984.95    5015503.08
        current_bio_latent             latent_chem__public_plus_kaggle          210180.68    5021738.97
current_probability_public                   gp_power1.0_rho1.0_median          212072.37    5034702.02
         current_bio_blend                              bio_weight_0.5          217883.21    5023845.70
         current_hurdle_gp          2.0_public_plus_kaggle_laten

## Paired public + Kaggle / Kaggle-only comparison

Same methods, parameters, competition validation rows and split roles. Public held-out molecule/scaffold overlap was excluded.

In [4]:
base = table[table.experiment.eq('current')].copy()
base['method'] = base.candidate.str.split('__').str[0]
print(base.pivot(index='method',columns='regime',values='mean_oof_mse').round(2).to_string())


regime                  kaggle_only  public_plus_kaggle
method                                                 
catboost_direct          5501602.81        1.038313e+08
catboost_log1p           5125011.44        5.135017e+06
chemeleon_domain         7973414.61        7.807106e+06
chemeleon_domain_log1p   5076402.90        5.061848e+06
chemeleon_mlp            9191511.41        6.286940e+07
chemeleon_mlp_log1p      5071066.32        8.404670e+06
chemeleon_ridge          7693582.04        5.600669e+07


## Verified official Kaggle Scores

Values are attached to exact submission bytes and API references. Unknown scores stay missing. Local endpoint-ratio diagnostics are intentionally absent from this report and from model selection.

In [5]:
print(table[table.official_kaggle_score.notna()][['experiment','candidate','mean_oof_mse','official_kaggle_score','submission_ref']].to_string(index=False))
registry = json.loads((root/'reports/current/kaggle_goal_submissions.json').read_text())
print('Current target:', registry['target'])
print('Submissions tracked across quota windows:', len(registry['submissions']))


                experiment                                  candidate  mean_oof_mse  official_kaggle_score  submission_ref
current_probability_public        gp_power2.0_rho1.0_observation_mean  4.973199e+06           1.855809e+07      58104352.0
      current_bio_transfer                        bio_fp__kaggle_only  5.014266e+06           1.900880e+06      58107726.0
                   current chemeleon_domain_log1p__public_plus_kaggle  5.061848e+06           1.586334e+06      58103006.0
              current_tail                           domain_cap_q0.99  5.061848e+06           1.586413e+06      58103262.0
         current_hurdle_gp                1.0_kaggle_only_latent_mean  5.070676e+06           1.949485e+06      58104456.0
Current target: top10 OR Score<800000
Submissions tracked across quota windows: 5


## Interpretation

A better raw OOF score can come from a tiny number of extreme structures while worsening zeros and ordinary SI. The failed mixed GP illustrates why this does not establish hidden-test generalization. Source shift and tail exposure are separate risks. Bootstrap intervals and detailed split/source checks are in `reports/current/protocol_audit.json`.

Official competition concentration columns never enter current fitting or inference. Public auxiliary IC50, EC50 and CC50 are separate tasks with independent provenance; censored observations impose directional bounds. Historic concentration-head scores are outside the current protocol.